In [1]:
import warnings
from dotenv import load_dotenv
import os
load_dotenv()

True

**Hugging Face LLM and Embed API connect**

In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

In [20]:
llm_endpoint = HuggingFaceEndpoint(repo_id=os.getenv('hf_model'))
hf_model = ChatHuggingFace(llm=llm_endpoint)

In [21]:
hf_emb = HuggingFaceEndpointEmbeddings(repo_id=os.getenv('hf_emb'))

**Tools and Agent**

In [45]:
from langchain_core.tools import tool, BaseTool, StructuredTool
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from typing import Annotated, Type
import requests

In [34]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import create_agent
import langchainhub

duck_search_tool = DuckDuckGoSearchRun()

In [101]:
class RudeusTimes(BaseModel):
    a : float = Field(required=True, description='a number to multiply with rudeus ratio')

# created pi times tool using basetool -> creates async version of tool.
class RudeusTimesTool(BaseTool):
    name:str = 'RudeusRatioTool'
    description: str = 'A tool to multiply number with rudeus ratio number'
    args_schema: type[BaseModel] = RudeusTimes

    def _run(self,a:float)->float:
        return 1.618033988 * a


rudeustime_basetool = RudeusTimesTool()

C:\Users\rudeu\AppData\Local\Temp\ipykernel_6080\4123173713.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a : float = Field(required=True, description='a number to multiply with rudeus ratio')


In [102]:
rudeustime_basetool.invoke({'a':1.1})

1.7798373868000001

In [103]:
messages = [HumanMessage(content='can you find rudues ratio times 1.1')]

In [104]:
hf_model_with_rudeustool = hf_model.bind_tools([rudeustime_basetool])
response = hf_model_with_rudeustool.invoke(messages)
messages.append(response)
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":1.1}', 'name': 'RudeusRatioTool', 'description': None}, 'id': 'call_chyfxid6l2snca2ht76bsfhr', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 220, 'total_tokens': 245}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff7e-6f09-7b03-b60b-5ee5f7dcfd7a-0', tool_calls=[{'name': 'RudeusRatioTool', 'args': {'a': 1.1}, 'id': 'call_chyfxid6l2snca2ht76bsfhr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 25, 'total_tokens': 245})

In [105]:
tool_response = rudeustime_basetool.invoke(response.tool_calls[0])
messages.append(tool_response)

In [106]:
hf_model_with_rudeustool.invoke(messages)

AIMessage(content="It seems like you're referring to a Rudeus ratio multiplied by 1.1, and the result is approximately 1.78. If you need more context or have any specific questions about this calculation, feel free to ask!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 102, 'total_tokens': 152}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fff7e-819b-7143-ac3d-05263bc5d14e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 102, 'output_tokens': 50, 'total_tokens': 152})

In [107]:
messages

[HumanMessage(content='can you find rudues ratio times 1.1', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":1.1}', 'name': 'RudeusRatioTool', 'description': None}, 'id': 'call_chyfxid6l2snca2ht76bsfhr', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 220, 'total_tokens': 245}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff7e-6f09-7b03-b60b-5ee5f7dcfd7a-0', tool_calls=[{'name': 'RudeusRatioTool', 'args': {'a': 1.1}, 'id': 'call_chyfxid6l2snca2ht76bsfhr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 25, 'total_tokens': 245}),
 ToolMessage(content='1.7798373868000001', name='RudeusRatioTool', tool_call_id='call_chyfxid6l2snca2ht76bsfhr')]

**AGENT CREATION**

In [ ]:
agent = create_agent(
    model=hf_model,
    tools=[rudeustime_basetool],
    system_prompt='you are an ReAct agent',
)

In [112]:
agent.invoke({
    'messages':[
        {
        'role':'user',
        'content':'find rudeus ratio of 7.3. and use the result and find its rudeus ratio'
        }
]})

{'messages': [HumanMessage(content='find rudeus ratio of 7.3. and use the result and find its rudeus ratio', additional_kwargs={}, response_metadata={}, id='87ef139e-4f68-4364-a584-5936aba1c2ab'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":7.3}', 'name': 'RudeusRatioTool', 'description': None}, 'id': 'call_nkqfpdhiy5izqyj48uv1abdj', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 220, 'total_tokens': 245}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff81-5e3b-78b2-aa4e-d99e9ca9ece3-0', tool_calls=[{'name': 'RudeusRatioTool', 'args': {'a': 7.3}, 'id': 'call_nkqfpdhiy5izqyj48uv1abdj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 25, 'total_tokens': 245}),
  ToolMessage(content='11.8116481124', name='RudeusRatioTool', id='b9f880b8-a483-4857-b0be-

In [ ]:
%pip install langgraph

langgraph._internal._pydantic.LangGraphOutput

In [ ]:
print('something')